In [11]:
from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root

from custom_rules import apply_rules

import os
import ast

In [12]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [13]:
df = read_dataset(data_path)

In [22]:
predictions = []
usages = []

In [23]:
for batch in tqdm(generator_list[:10]):
    response = test_run(batch, prompt_path="prompt-ds-main-classifier.txt")
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 10/10 [02:02<00:00, 12.24s/it]


In [24]:
flat = [x for sub in predictions for x in sub]

In [25]:
ground_truths = []

In [26]:
for i in range(len(flat) // len(predictions)):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [27]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [28]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

array([[45, 17],
       [ 4, 34]])

In [29]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

0.79

In [30]:
comparison

,Predicted,Ground Truth,Text
0,1,1,Imagine blaming this on Israel instead of Hamas
1,1,1,Blinken is the Butcher of Blinken ButcherOfGaz...
2,1,0,"And as far as your gun control, Chicago knows ..."
3,0,0,All the spoilers are on Fox News daily.
4,1,1,Are red states stealing from California?
...,...,...,...
95,0,0,Israel at war after Hamas launches surprise at...
96,0,1,Well see how this story ends. Theres a conside...
97,1,0,Texas Republicans Target Climate Science in Te...
98,0,0,US seeks Israeli approval to arm PA troops in ...


In [34]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong

,Predicted,Ground Truth,Text
2,1,0,"And as far as your gun control, Chicago knows ..."
14,0,1,And they will call it election interference an...
24,1,0,They said voter fraud doesnt exist tho...
26,0,1,Germany is deporting Syrian and Afghan refugee...
33,1,0,Israel is attempting genocide and the U.S. is ...
38,0,1,"Microphones for imperialism, yet again."
40,1,0,"CSG holds meeting, students speak on IsraelHam..."
42,1,0,theyre not taking down the Iron dome. ffs people.
43,1,0,Calling for Impeachment. ASAP. impeachmentnow ...
44,1,0,Israel closes border crossing to Gaza workers ...


In [29]:
submission = create_submission(df, flat)

In [30]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_eng.csv'), index=False)